In [2]:
# importing libraries
import asyncio
import httpx 

In [3]:
# Semaphore handling maximum requests to prevent from waiting in queue.

# Setting configuration
CONCURRANCY_LIMIT = 3
QUEUE_TIMEOUT = 5.0

# Worker industry grade 
async def protected_ai_worker(client, url, sem, task_id):
    try:
        async with asyncio.timeout(QUEUE_TIMEOUT):
            async with sem:
                print(f"Slot Acquired for Task {task_id} entering critical zone")
                response = await client.get(url)
                print(f"Slot Released for Task {task_id} with status code: {response.status_code}")
                return (url, response.status_code)
    except TimeoutError:
        print(f"TIMEOUT: Task {task_id} dropped becuase it spent <{QUEUE_TIMEOUT}s")
        return {url , f"Error: time out for queuing request"}

    except Exception as err:
        print(f"FAILURE: Task {task_id} encountered unexpected erro: {err}")
        return {url, f"Error: {str(err)}"}

async def main():
    # instantiate global safe guard
    api_semaphore = asyncio.Semaphore(CONCURRANCY_LIMIT)
    target_apis = ['https://httpbin.org/get'] * 15

    async with httpx.AsyncClient() as client:
        tasks = (
            protected_ai_worker(client, url, api_semaphore, idx)
            for idx, url in enumerate(target_apis, start=1)
        )
        results = await asyncio.gather(*tasks, return_exceptions=True)

        print("Production pipeline execution summary")
        for url, status in results:
            print(f"Target: {url} -> Resolution: {status}")

await main()

Slot Acquired for Task 1 entering critical zone
Slot Acquired for Task 2 entering critical zone
Slot Acquired for Task 3 entering critical zone
Slot Released for Task 1 with status code: 200
Slot Acquired for Task 4 entering critical zone
Slot Released for Task 2 with status code: 200
Slot Acquired for Task 5 entering critical zone
Slot Released for Task 4 with status code: 200
Slot Released for Task 3 with status code: 200
Slot Acquired for Task 6 entering critical zone
Slot Acquired for Task 7 entering critical zone
Slot Released for Task 5 with status code: 200
Slot Acquired for Task 8 entering critical zone
Slot Released for Task 7 with status code: 200
Slot Released for Task 6 with status code: 200
Slot Acquired for Task 9 entering critical zone
Slot Acquired for Task 10 entering critical zone
Slot Released for Task 8 with status code: 200
Slot Acquired for Task 11 entering critical zone
Slot Released for Task 9 with status code: 200
Slot Released for Task 11 with status code: 200

In [11]:
# GLOBAL INITIALIZATION
CONCURRANCY_LIMIT = 3
API_TIMEOUT = 3.0
QUEUE_TIMEOUT = 10

# worker function
async def fetch_api_worker(client, url, sem, task_id):
    try:
        async with asyncio.timeout(QUEUE_TIMEOUT):
            async with sem:
                print(f"[SLOT ACQUIRED] Task {task_id} is entering network zone.")
                await asyncio.sleep(1.5)
                response = await client.get(url)
                print(f"[SLOT RELEASED] Task {task_id} is SUCCEDED {response.status_code}")
                return (url, response.status_code)
    except TimeoutError:
        print(f"TIMEOUT ERROR: Task {task_id} dropped becuase it spent {QUEUE_TIMEOUT}s")
        return {url, f"Error: Time out Error"}
    except Exception as err:
        print(f"FAILURE: Task {task_id} encountered unexpected error {err}")
        return {url, f"Error: {str(err)}"}

async def main():
    api_semaphore = asyncio.Semaphore(CONCURRANCY_LIMIT)
    target_url = ['https://httpbin.org/get']*10

    timeout = httpx.Timeout(API_TIMEOUT)
    # single client pool
    async with httpx.AsyncClient(timeout=timeout) as client:

        tasks = [
            fetch_api_worker(client, url, api_semaphore, idx)
            for idx, url in enumerate(target_url, start=1)

        ]

        results = await asyncio.gather(*tasks, return_exceptions=True)

        for result in results:
            if isinstance(result, BaseException):
                print(f"Request failed: {result}")
                continue
            url, status = result
            print(f"Target: {url} , Resolution: {status}")

await main()






    

[SLOT ACQUIRED] Task 1 is entering network zone.
[SLOT ACQUIRED] Task 2 is entering network zone.
[SLOT ACQUIRED] Task 3 is entering network zone.
FAILURE: Task 1 encountered unexpected error 
FAILURE: Task 2 encountered unexpected error 
FAILURE: Task 3 encountered unexpected error 
[SLOT ACQUIRED] Task 4 is entering network zone.
[SLOT ACQUIRED] Task 5 is entering network zone.
[SLOT ACQUIRED] Task 6 is entering network zone.
[SLOT RELEASED] Task 6 is SUCCEDED 200
[SLOT RELEASED] Task 4 is SUCCEDED 200
[SLOT RELEASED] Task 5 is SUCCEDED 200
[SLOT ACQUIRED] Task 7 is entering network zone.
[SLOT ACQUIRED] Task 8 is entering network zone.
[SLOT ACQUIRED] Task 9 is entering network zone.
[SLOT RELEASED] Task 9 is SUCCEDED 200
[SLOT RELEASED] Task 8 is SUCCEDED 200
[SLOT ACQUIRED] Task 10 is entering network zone.
[SLOT RELEASED] Task 7 is SUCCEDED 200
TIMEOUT ERROR: Task 10 dropped becuase it spent 10s
Target: Error:  , Resolution: https://httpbin.org/get
Target: Error:  , Resolution: h

In [6]:
# Async generator 

async def stream_llm_response():
    words = ["Applied", "AI", "Engineering", "with", "Asyncio", "is", "Amazing!"]
    for word in words:
        await asyncio.sleep(0.2)
        yield word

async def main():
    print("Stream token generation is started live")

    async for token in stream_llm_response():
        print(token + "", end=" ", flush=True)

await main()


Stream token generation is started live
Applied AI Engineering with Asyncio is Amazing! 

In [5]:
async def stream_llm_tokens(user_prompt):
    words = f"AI is reponse '{user_prompt}': Retrieval Augmented Generation is currently widely envolved in production companies to fetch latest information from external resources.".split()
    for token in words:
        await asyncio.sleep(0.3)
        yield token

async def main():
    print('Stream token generation is started live generate tokens.')

    async for word in stream_llm_tokens('What is RAG'):
        print(word + " ", end = " ", flush=True)
        # print()     # If you want to print each word in new line you uncomment this
await main()

Stream token generation is started live generate tokens.
AI  is  reponse  'What  is  RAG':  Retrieval  Augmented  Generation  is  currently  widely  envolved  in  production  companies  to  fetch  latest  information  from  external  resources.  

## Queue concept in asyncio.
- It is useful to prevent the error of 429 (Too Many Request) when ingestion any documents or put/retrive prompts/answer unlimit users at a time.
* `Queue has 4 methods:`
- 1. `queue.put(itme):` useful to put item/process documents in vector database, if database limit already hit it waits until any item remove and space is empty.
- 2. `queue.get(item):` Useful to get item from database in case database is empty no recorded stored in db it wait until any record put.
- 3. `queue.task_done():` useful and mandatory to at the end of queue.get(item) tells queue to no more wait any getting or putting record task is done completely.
- 4. `queue.join():` wait and stop program until in main pipeline all records/items are get completely  and task_done .


In [2]:
import asyncio
import random

async def ai_processor_worker(task_queue , task_id):
    while True:
        incoming_prompt = await task_queue.get()
        print(f"Worker {task_id} getting {incoming_prompt}")

        try:
            await asyncio.sleep(random.uniform(0.5 , 1.5))
            print(f"Worker {task_id} get successfully response of {incoming_prompt}")
        except httpx.HTTPError as e:
            print(f"Worker {task_id} get Network Error {e}")

        finally:
            task_queue.task_done()

async def main():
    my_queue = asyncio.Queue(maxsize=4)
    incoming_user_prompts = [
        "Write a python script for RAG",
        "Explain fine-tuning to a 5 year old",
        "Fix my recursion error code",
        "Generate a vector embedding matrix"
    ]

    acitive_workers = [
        asyncio.create_task(ai_processor_worker(my_queue, 1)),
        asyncio.create_task(ai_processor_worker(my_queue, 2))
    ]
    for prompt in incoming_user_prompts:
        await my_queue.put(prompt)

    print(f'\nAll prompt queued up now spliting safely')
    await my_queue.join()

    for task in acitive_workers:
        task.cancel()

await main()




All prompt queued up now spliting safely
Worker 1 getting Write a python script for RAG
Worker 2 getting Explain fine-tuning to a 5 year old
Worker 1 get successfully response of Write a python script for RAG
Worker 1 getting Fix my recursion error code
Worker 2 get successfully response of Explain fine-tuning to a 5 year old
Worker 2 getting Generate a vector embedding matrix
Worker 1 get successfully response of Fix my recursion error code
Worker 2 get successfully response of Generate a vector embedding matrix


In [12]:
import random
import asyncio
import logging
import httpx

# 1. Config logging
logging.basicConfig(
    filename=('link_log_error.log'),
    level=logging.INFO, 
    format="%(levelname)s - %(message)s",
    )
logger = logging.getLogger('FetchLogError')

# 2. Store links (let's pretend one is broken)
links = [
    "https://httpbin.org/get", 
    "https://broken-link-xyz.com", 
    "https://httpbin.org/get"
]

async def fetch_url(client, url, idx):
    try:
        logger.info(f"Worker is started to fetching url")
        response = await client.get(url)
        await asyncio.sleep(random.uniform(0.5, 1.5))
        response.raise_for_status()
        logger.info(f"Fetched {idx} url successfully.")
        return (url, f"Success: {response.status_code}")
    except httpx.HTTPError as e:
        logger.critical(f"'{url}' is broken please correct this, The Error is '{e}'")
        return (url, f"Failed: {str(e)}")
async def main():
    async with httpx.AsyncClient() as client:
        async with asyncio.TaskGroup() as tg:
            tasks = [tg.create_task(fetch_url(client=client, url=url, idx=idx))
                        for idx, url in enumerate(links, start=1)
                ]

        for result in tasks:
            print(result.result())
await main()

('https://httpbin.org/get', 'Success: 200')
('https://broken-link-xyz.com', 'Failed: [Errno -2] Name or service not known')
('https://httpbin.org/get', 'Success: 200')


In [16]:
# global initialization
import httpx

CONCURRANCY_LLM_LIMIT = 3

MAX_ALLOWED_TCP_SOCKETS = 50
MAX_KEEPALIVE_CONNECTS = 20

async def embed_document_chunk_worker(client, chunks, sem, idx):
    async with sem:
        try:
            print(f"Processing Chunk {idx} entering network loop")
            await asyncio.sleep(random.uniform(0.5, 1.5))
            response = await client.post(
                'https://openai.com/post',
                json = {'input': chunks, 'model': 'gpt_3_small_embed'}
            )
            response.raise_for_status()
            print(f"Chunks created successfully with status code: {response.status_code}")
            return (idx, 'Success')
        except Exception as e:
            return (idx, f"Failed {str(e)}")

async def main():
    rate_limiter = asyncio.Semaphore(CONCURRANCY_LLM_LIMIT)

    network_limits = httpx.Limits(
        max_connections=MAX_ALLOWED_TCP_SOCKETS,
        max_keepalive_connections= MAX_KEEPALIVE_CONNECTS)

    raw_documents_chunks = raw_document_chunks = ["Sample text payload chunk string"] * 100

    async with httpx.AsyncClient(limits=network_limits) as production_client:
        tasks = [
            embed_document_chunk_worker(production_client, chunk, rate_limiter, idx)
            for idx, chunk in enumerate(raw_document_chunks, start=1)
        ]
        result = await asyncio.gather(*tasks, return_exceptions=True)
        print(f"All chunks successfully embeded {result}")

await main()
    


Processing Chunk 1 entering network loop
Processing Chunk 2 entering network loop
Processing Chunk 3 entering network loop
Processing Chunk 4 entering network loop
Processing Chunk 5 entering network loop
Processing Chunk 6 entering network loop
Processing Chunk 7 entering network loop
Processing Chunk 8 entering network loop
Processing Chunk 9 entering network loop
Processing Chunk 10 entering network loop
Processing Chunk 11 entering network loop
Processing Chunk 12 entering network loop
Processing Chunk 13 entering network loop
Processing Chunk 14 entering network loop
Processing Chunk 15 entering network loop
Processing Chunk 16 entering network loop
Processing Chunk 17 entering network loop
Processing Chunk 18 entering network loop
Processing Chunk 19 entering network loop
Processing Chunk 20 entering network loop
Processing Chunk 21 entering network loop
Processing Chunk 22 entering network loop
Processing Chunk 23 entering network loop
Processing Chunk 24 entering network loop
P

In [4]:
# to_thread function instead of direct with open and json.dumps 
import time
def potatos(file_name, text):
    print(f"Helper function started cutting potatos in {file_name}")
    with open(file_name, 'w', encoding='utf-8') as f:
        f.write(text)
    time.sleep(0.3)
    print(f"Helper function cut all potatos he Finish his work.")
    return "Bowl of cut potatos"

async def main():
    print(f"Me: Asking to Helper to do this work in background.")
    result = await asyncio.to_thread(potatos, 'text.txt', 'Hello word')
    print(f"I take result {result}")

await main()

Me: Asking to Helper to do this work in background.
Helper function started cutting potatos in text.txt
Helper function cut all potatos he Finish his work.
I take result Bowl of cut potatos


In [11]:
import asyncio
import httpx
import logging
import json
from pathlib import Path

# basics configuration
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt= "%H:%M:%S"
)
logger = logging.getLogger("ProcessDocument")

class ProductionDocumentProcessor:
    def __init__(self , storage_dir):
        self.storage_dir = storage_dir
        self.storage_dir.mkdir(parents = True, exist_ok = True)

    @staticmethod
    def _heavy_cpu_token_count(text):
        total_characters = len(text)
        aprox_tokens = total_characters//5
        return aprox_tokens
    @staticmethod
    def _blocking_disk_save(file_path, data):
        with open(file_path, 'w', encoding="utf-8") as f:
            json.dump(data, f, indent=4)

    async def process_document(self, doc_id , raw_text):
        logger.info(f"Processor Document {doc_id} is on main event loop")
        token_count = await asyncio.to_thread(self._heavy_cpu_token_count, raw_text)
        logger.info(f"Process Document {doc_id} has approximates {token_count} Tokens.")

        payload = {
            'doc_id' : doc_id,
            'token_count' : token_count,
            'raw_text' : raw_text
        }

        output_file = self.storage_dir / f"{doc_id}.json"
        await asyncio.to_thread(self._blocking_disk_save, output_file, payload)

        return payload

async def incoming_requests(service):
    docs = [
    ("doc_101", "Artificial intelligence systems require low-latency non-blocking I/O."),
    ("doc_102", "Vector databases store dense embeddings for semantic search retrieval."),
    ("doc_103", "Pydantic V2 uses a fast Rust core for strict schema validation.")
]
    tasks = [
        service.process_document(doc_id, raw_text)
        for doc_id , raw_text in docs
    ]

    results = await asyncio.gather(*tasks, return_exceptions=True)
    logger.info(f"All documents {len(results)} have successfully processd.")

async def main():
    storage = Path("process_data")
    processor = ProductionDocumentProcessor(storage)

    await incoming_requests(processor)


await main()










10:39:59 [INFO] Processor Document doc_101 is on main event loop
10:39:59 [INFO] Processor Document doc_102 is on main event loop
10:39:59 [INFO] Processor Document doc_103 is on main event loop
10:39:59 [INFO] Process Document doc_101 has approximates 13 Tokens.
10:39:59 [INFO] Process Document doc_102 has approximates 14 Tokens.
10:39:59 [INFO] Process Document doc_103 has approximates 12 Tokens.
10:39:59 [INFO] All documents 3 have successfully processd.
